# 06 — Social & web charts

Publication-ready renders of the charts the owner selected in `04-viz`. Two targets per chart
(the standard two-target web system):

- **Social** → `outputs/social/` — full chrome (title, subtitle, source, watermark), `twitter_landscape`.
- **Web** → `outputs/web/` — `web_mode=True` drops title/subtitle/source (the Pages markdown supplies
  them), keeps the watermark, hands the freed space to the data. `web` preset (1664 wide).

**Selected for social posting:** most-unequal side-by-side (C1), most-equal side-by-side (C4),
wealth-vs-inequality income scatter (C6), consumption scatter (C7), life-expectancy catastrophes
(annotated small multiples), the US metric dashboard, and the US world-rank bars.

**Web-only (not posted, but must render cleanly for the page):** the three browse grids
(fertility / life expectancy / unemployment small multiples).

All charts come from the SHARED factory templates (same as `04-viz`), so what we explored is what
ships. The tall small-multiples/dashboard/catastrophe charts have variable height, so we render and
save them directly (preserving natural height) rather than through the preset-resizing exporter.

> Rendering engine note: matplotlib is unusable on this project's Python 3.14 venv; everything is
> the shared Pillow factory. DuckDB opened read-only, closed in Cleanup.

In [ ]:
import sys, os
from pathlib import Path
import duckdb, pandas as pd

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT.parent.parent / 'shared'))

import chart_templates as ct
from colors import c
from viz import PRESETS
from IPython.display import Image as IPImage, display as ipy_display

con = duckdb.connect('data/project.duckdb', read_only=True)

SOCIAL_DIR = Path('outputs/social'); SOCIAL_DIR.mkdir(parents=True, exist_ok=True)
WEB_DIR = Path('outputs/web'); WEB_DIR.mkdir(parents=True, exist_ok=True)
SW, SH, _ = PRESETS['twitter_landscape']   # 1600x900 social
WW, WH, _ = PRESETS['web']                  # 1664x936 web
SRC = 'World Bank WDI + PIP (survey years vary)  ·  @unwelcomedata'
INCOME_COLOR, CONSUMPTION_COLOR, NAVY = c('gold'), c('teal'), c('navy')

def export(img, name, target):
    d = SOCIAL_DIR if target == 'social' else WEB_DIR
    out = d / f'{name}.png'
    img.save(out, format='PNG', optimize=True)
    print(f'  {target:6s} -> {out}  ({img.size[0]}x{img.size[1]})')
    return out

def show(img):
    import io
    b = io.BytesIO(); img.save(b, format='PNG'); b.seek(0)
    ipy_display(IPImage(data=b.getvalue()))

print('ready')

## Data

The latest snapshot (for the ranking/scatter charts) and the annual panel (for the time-series
small multiples, catastrophe panels, and US dashboard).

In [ ]:
snap = pd.read_parquet('data/processed/countries_latest.parquet')
g = snap.dropna(subset=['gini_index']).copy()
income = g[g.gini_welfare_type == 'income'].copy()
consumption = g[g.gini_welfare_type == 'consumption'].copy()

panel = con.execute('SELECT iso_alpha3, country_name, year, fertility_rate, life_expectancy, '
                    'unemployment_pct FROM countries_clean').df()
panel_full = con.execute('SELECT iso_alpha3, country_name, year, fertility_rate, life_expectancy, '
                         'unemployment_pct, gdp_per_capita_ppp, gini_index, urban_pct '
                         'FROM countries_clean').df()
us_only = panel_full[panel_full.iso_alpha3 == 'USA'].copy()
print(f'{len(g)} countries with Gini; panel {len(panel):,} rows')

Shared subtitle lead — every Gini chart states what the number means (0 = everyone equal, 100 = one
person has it all). Web charts drop the subtitle, so the **page markdown must carry the Gini
definition + the metric caveat** above these charts (info-preservation).

In [ ]:
GINI = 'Gini index 0–100: 0 = everyone equal, 100 = one person has it all (higher = more unequal). '

def prep_rank(df, n, ascending=False):
    d = (df.nsmallest(n, 'gini_index') if ascending else df.nlargest(n, 'gini_index')).copy()
    if ascending: d = d.sort_values('gini_index', ascending=False)
    d['label'] = d['gini_index'].map(lambda v: f'{v:.1f}')
    return d

## C1 — Most unequal: consumption vs income (side-by-side)

In [ ]:
cons_top, inc_top = prep_rank(consumption, 12), prep_rank(income, 12)
def render_c1(web):
    return ct.side_by_side_bars(
        df_left=cons_top, df_right=inc_top,
        category_col='country_name', value_col='gini_index', total_label_col='label',
        left_title='Consumption-based', right_title='Income-based',
        left_color=CONSUMPTION_COLOR, right_color=INCOME_COLOR,
        title='Most unequal countries — by survey type, shown separately',
        subtitle=(GINI + 'Consumption surveys (lower-income) vs income surveys (richer) are not comparable — not merged.'),
        source=SRC,
        img_width=(WW if web else SW), img_height=(WH if web else SH), web_mode=web)
export(render_c1(False), 'c1_most_unequal_by_metric', 'social')
export(render_c1(True), 'c1_most_unequal_by_metric', 'web')
show(render_c1(False))

## C4 — Most equal: consumption vs income (side-by-side)

In [ ]:
cons_eq, inc_eq = prep_rank(consumption, 12, ascending=True), prep_rank(income, 12, ascending=True)
def render_c4(web):
    return ct.side_by_side_bars(
        df_left=cons_eq, df_right=inc_eq,
        category_col='country_name', value_col='gini_index', total_label_col='label',
        left_title='Consumption-based', right_title='Income-based',
        left_color=CONSUMPTION_COLOR, right_color=INCOME_COLOR,
        title='Most equal countries — by survey type, shown separately',
        subtitle=(GINI + 'Lower = more equal. India’s low consumption value partly reflects the metric, not only equality.'),
        source=SRC,
        img_width=(WW if web else SW), img_height=(WH if web else SH), web_mode=web)
export(render_c4(False), 'c4_most_equal_by_metric', 'social')
export(render_c4(True), 'c4_most_equal_by_metric', 'web')
show(render_c4(False))

## C6 — Wealth vs inequality, income-based (scatter, loner-labeled)

In [ ]:
inc_gdp = income[income.gdp_per_capita_ppp.notna()].copy()
def render_c6(web):
    return ct.scatter_plot(
        df=inc_gdp, x_col='gdp_per_capita_ppp', y_col='gini_index', label_col='country_name',
        label_loners=8, x_axis_label='GDP per capita, PPP (current int$)', y_axis_label='Gini index (0–100)',
        point_color=INCOME_COLOR, x_fmt=(lambda v: f'${v/1000:,.0f}k'),
        title='Wealth vs inequality — income-based countries',
        subtitle=(GINI + 'Richer income-survey countries are clearly more equal (r ≈ −0.51). Income metric only.'),
        source=SRC, img_width=(WW if web else SW), img_height=(WH if web else SH), web_mode=web)
export(render_c6(False), 'c6_wealth_vs_inequality_income', 'social')
export(render_c6(True), 'c6_wealth_vs_inequality_income', 'web')
show(render_c6(False))

## C7 — Wealth vs inequality, consumption-based (scatter, loner-labeled)

In [ ]:
cons_gdp = consumption[consumption.gdp_per_capita_ppp.notna()].copy()
def render_c7(web):
    return ct.scatter_plot(
        df=cons_gdp, x_col='gdp_per_capita_ppp', y_col='gini_index', label_col='country_name',
        label_loners=8, x_axis_label='GDP per capita, PPP (current int$)', y_axis_label='Gini index (0–100)',
        point_color=CONSUMPTION_COLOR, x_fmt=(lambda v: f'${v/1000:,.0f}k'),
        title='Wealth vs inequality — consumption-based countries',
        subtitle=(GINI + 'Weaker link among consumption-survey countries (r ≈ −0.23). Consumption metric only.'),
        source=SRC, img_width=(WW if web else SW), img_height=(WH if web else SH), web_mode=web)
export(render_c7(False), 'c7_wealth_vs_inequality_consumption', 'social')
export(render_c7(True), 'c7_wealth_vs_inequality_consumption', 'web')
show(render_c7(False))

## Catastrophe — life-expectancy shocks (annotated small multiples)

Variable height (grows with panel count), so exported at natural size. CAR flagged ⚠ verify.

In [ ]:
catastrophe_specs = [
    {'entity': 'Cambodia',                 'name': 'Cambodia',                 'event': 'Khmer Rouge',         'mark_x': 1977},
    {'entity': 'Rwanda',                   'name': 'Rwanda',                   'event': 'Genocide',            'mark_x': 1994},
    {'entity': 'Timor-Leste',              'name': 'Timor-Leste',              'event': 'Invasion/occupation', 'mark_x': 1978},
    {'entity': 'Central African Republic', 'name': 'Central African Republic', 'event': 'Conflict',            'mark_x': 2009, 'flag': 'verify'},
]
def render_cat(web, width):
    return ct.annotated_small_multiples(
        df=panel, entity_col='country_name', x_col='year', value_col='life_expectancy',
        specs=catastrophe_specs, line_color=NAVY,
        title='When catastrophe collapses life expectancy',
        subtitle='Life expectancy at birth (years). Each panel is one country on its own scale; the marked year is the low point.',
        source=SRC, img_width=width, web_mode=web)
export(render_cat(False, SW), 'catastrophe_life_expectancy', 'social')
export(render_cat(True, WW), 'catastrophe_life_expectancy', 'web')
show(render_cat(False, SW))

## US metric dashboard (metric_dashboard) — the reusable entity-profile chart

0-based panels; fact-based annotations only (COVID, 2008 crisis, 2002 Gini survey break).

In [ ]:
us_metrics = [
    {'col': 'life_expectancy',    'name': 'Life expectancy',   'unit': 'years',        'annot': [(2021, 'COVID')]},
    {'col': 'gdp_per_capita_ppp', 'name': 'GDP per capita',    'unit': 'PPP int$'},
    {'col': 'gini_index',         'name': 'Income inequality', 'unit': 'Gini (income-based)',
     'breaks': [(2002, 'survey redesign — pre/post not comparable')]},
    {'col': 'fertility_rate',     'name': 'Fertility rate',    'unit': 'births/woman'},
    {'col': 'unemployment_pct',   'name': 'Unemployment',      'unit': '% labor force',
     'annot': [(2010, '9.6% — post-2008 crisis'), (2020, '8.1% — COVID')]},
    {'col': 'urban_pct',          'name': 'Urban population',   'unit': '% of total'},
]
def render_dash(web, width):
    return ct.metric_dashboard(
        df=us_only, metrics=us_metrics, line_color=NAVY,
        title='United States — the key metrics over time',
        subtitle='US only. Each panel is 0-based (true scale, not zoomed). Gini is income-based.',
        source=SRC, img_width=width, web_mode=web)
export(render_dash(False, SW), 'us_dashboard', 'social')
export(render_dash(True, WW), 'us_dashboard', 'web')
show(render_dash(False, SW))

## Where the US ranks in the world (percentile bars)

Gini ranked vs income-based countries only (never mixing survey types).

In [ ]:
income_pool = snap[snap['gini_welfare_type'] == 'income']
def us_pct(col, pool=None):
    s = (pool if pool is not None else snap)[col].dropna()
    usv = snap.loc[snap.iso_alpha2 == 'US', col]
    if usv.empty or pd.isna(usv.iloc[0]): return None, None
    usv = usv.iloc[0]
    return round(100.0 * (s <= usv).mean(), 0), usv
rows = []
for col, lbl, pool in [('gdp_per_capita_ppp','GDP per capita (PPP)', None),
                       ('gini_index','Income inequality (Gini, income-based)', income_pool),
                       ('life_expectancy','Life expectancy', None),
                       ('fertility_rate','Fertility rate', None),
                       ('unemployment_pct','Unemployment', None)]:
    p, _ = us_pct(col, pool)
    rows.append({'indicator': lbl, 'pct': p, 'label': f'{p:.0f}th pct'})
us_rank = pd.DataFrame(rows).sort_values('pct', ascending=False)
def render_rank(web):
    return ct.single_ranked_bars(
        df=us_rank, category_col='indicator', value_col='pct', total_label_col='label',
        bar_color=NAVY,
        title='Where the United States ranks in the world',
        subtitle='US global percentile per metric (100th = highest, 0th = lowest). Gini vs income-based countries only.',
        source=SRC, img_width=(WW if web else SW), img_height=(WH if web else SH), web_mode=web)
export(render_rank(False), 'us_world_rank', 'social')
export(render_rank(True), 'us_world_rank', 'web')
show(render_rank(False))

## Web-only browse grids (not posted)

The three small-multiples grids. These are **web-only** — too big/detailed for a social card, but a
good "browse everything" artifact for the page. Rendered at web width only (natural tall height).

In [ ]:
browse = [
    ('fertility_rate', 'births/woman', 'Fertility rate over time — births per woman, by country', 'browse_fertility'),
    ('life_expectancy', 'years', 'Life expectancy over time — years at birth, by country', 'browse_life_expectancy'),
    ('unemployment_pct', '%', 'Unemployment rate over time — % of labor force (ILO estimate), by country', 'browse_unemployment'),
]
for col, unit, title, name in browse:
    img = ct.small_multiples_grid(
        df=panel, entity_col='country_name', x_col='year', value_col=col,
        title=title, subtitle='One panel per country, shared y-axis. Corner number = latest value.',
        source=SRC, unit=unit, line_color=NAVY, img_width=WW, web_mode=True)
    export(img, name, 'web')
print('browse grids done (web only)')

## Verify the social/web output set

In [ ]:
print('social:', sorted(p.name for p in SOCIAL_DIR.glob('*.png')))
print('web:   ', sorted(p.name for p in WEB_DIR.glob('*.png')))

## Cleanup

In [ ]:
con.close()
print('Connection closed.')